# Memory
# 0. 介绍

**研究背景**：大模型的单次请求本身没有跨会话记忆。Agent 要在后续任务中继续使用用户偏好、任务进度和已确认事实，外层程序就必须决定记住什么、存在哪里、何时取回，以及信息变化后如何更新或忘记。

**现存问题**：生产中常见的错误基线是保存全部历史，再只按文本相似度取回最接近的内容。这样会把过期偏好、已被替代的规则或其他会话的数据重新放进当前上下文；请求虽然能够正常完成，大模型却可能依据错误记忆行动。历史越多，冲突、隐私泄露和无关内容占用上下文的问题越严重。

**解决方案**：本 Notebook 将实现一个极简的 Memory，采用`结构化记忆生命周期 + 分层检索 + 来源与失效管理`机制：只保存值得跨会话保留的信息，为每条记录附上作用域、来源、时间和有效状态，并按能力递进实现四种检索方式：`Glob` 用路径或标签模式快速筛选明确目标，`BM25` 按关键词匹配强度排序，`嵌入向量相似度`按语义接近程度召回，`混合检索`合并关键词与语义结果，最后用`重排序`对候选记录进行更精细的相关性判断。检索前先隔离用户与会话并排除失效记录，再把少量高分结果注入当前上下文。后文会分别介绍每种方式的简单原理和具体运行方式，并在同一查询、同一记忆库与同一成功标准下比较召回结果、任务正确性、Token、成本和延迟；同时用真实 API 对比基线版本召回过期记忆而失败，与改进版本处理新旧冲突后只召回当前有效事实而成功，从而直观看到可靠记忆的关键不是“存得更多”，而是“在正确范围内取回当前可信的信息”。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 准备有冲突的记忆
记忆系统最危险的问题不是完全找不到信息，而是把已经过期的信息当成当前事实。下面准备四条记录：同一用户的一条旧周报偏好和一条新周报偏好、一条无关偏好，以及另一名用户的相似偏好。路径用于 Glob 筛选，正文用于后续的关键词和语义检索。

In [2]:
# path 让 Glob 可以按用户和记忆类型筛选候选记录
# updated_at 与 status 说明一条记忆现在是否仍然有效
memory_records = [
    {
        "id": "M-001",
        "owner": "user-a",
        "path": "memory/user-a/preferences/report-2025-11.md",
        "content": "当前周报偏好：使用表格，详细列出进展、风险和下一步。",
        "updated_at": "2025-11-03",
        "status": "superseded",
    },
    {
        "id": "M-002",
        "owner": "user-a",
        "path": "memory/user-a/preferences/report-2026-07.md",
        "content": "后来确认：汇报只保留三个简短要点，不要表格。",
        "updated_at": "2026-07-18",
        "status": "active",
    },
    {
        "id": "M-003",
        "owner": "user-a",
        "path": "memory/user-a/preferences/meeting-2026-06.md",
        "content": "会议纪要需要记录参会人和决定事项。",
        "updated_at": "2026-06-09",
        "status": "active",
    },
    {
        "id": "M-004",
        "owner": "user-b",
        "path": "memory/user-b/preferences/report-2026-07.md",
        "content": "当前周报偏好：使用表格展示所有数据。",
        "updated_at": "2026-07-20",
        "status": "active",
    },
]

for memory in memory_records:
    print(memory["id"], "|", memory["owner"], "|", memory["status"], "|", memory["content"])

M-001 | user-a | superseded | 当前周报偏好：使用表格，详细列出进展、风险和下一步。
M-002 | user-a | active | 后来确认：汇报只保留三个简短要点，不要表格。
M-003 | user-a | active | 会议纪要需要记录参会人和决定事项。
M-004 | user-b | active | 当前周报偏好：使用表格展示所有数据。


输出显示了后续实验共用的四条记忆。`M-001` 与 `M-002` 属于同一用户但内容冲突，只有较新的 `M-002` 仍然有效；`M-003` 与任务无关，`M-004` 则属于另一名用户。下一步固定五种检索方式及其分工。

## 2.2 固定检索路线
五种方法不是简单地重复同一件事。Glob 先缩小文件范围，BM25 和嵌入向量分别从关键词与语义角度召回，混合检索合并两种信号，重排序最后对少量候选做更精细的判断。下面先固定名称和简单原理，后文再逐个实现。

In [3]:
# order 表示真实运行时从候选筛选到最终排序的先后顺序
# principle 只保留每种方法最核心、最容易观察的作用
retrieval_methods = [
    {"order": 1, "name": "Glob", "principle": "用路径通配模式筛选候选文件"},
    {"order": 2, "name": "BM25", "principle": "按关键词匹配强度和词语稀有程度排序"},
    {"order": 3, "name": "嵌入向量相似度", "principle": "把查询和记忆变成向量，按语义距离排序"},
    {"order": 4, "name": "混合检索", "principle": "合并关键词排序与语义排序"},
    {"order": 5, "name": "重排序", "principle": "结合查询与记录状态重新排列少量候选"},
]

for method in retrieval_methods:
    print(method["order"], method["name"], "-", method["principle"])

1 Glob - 用路径通配模式筛选候选文件
2 BM25 - 按关键词匹配强度和词语稀有程度排序
3 嵌入向量相似度 - 把查询和记忆变成向量，按语义距离排序
4 混合检索 - 合并关键词排序与语义排序
5 重排序 - 结合查询与记录状态重新排列少量候选


输出给出了完整检索链。前四步负责找到候选，最后一步负责在候选中选出最适合当前任务的记录；后面的效果对比会始终使用同一份记忆库。下一步写出唯一的用户任务。

## 2.3 写出具体任务
任务故意只说“现在的偏好”，不在请求中重复正确格式。Agent 必须从跨会话记忆中找出当前有效记录，再整理三条已经给定的工作进展。

In [4]:
# current_user 是检索作用域，不依赖大模型猜测用户身份
# facts 固定了所有检索方式都要整理的相同任务内容
current_user = "user-a"
facts = ["接口开发完成", "自动测试通过", "使用文档待补"]
user_query = "请按我现在的周报偏好整理这三项进展。"

print("当前用户：", current_user)
print("用户请求：", user_query)
print("待整理内容：", facts)

当前用户： user-a
用户请求： 请按我现在的周报偏好整理这三项进展。
待整理内容： ['接口开发完成', '自动测试通过', '使用文档待补']


输出显示当前任务属于 `user-a`，但用户没有再次说明周报格式。这正是 Memory 要解决的问题：从历史记录中找回当前有效偏好，而不是让模型猜测。下一步定义模型必须提交的结构化结果。

## 2.4 定义模型输出格式
为了直接比较不同检索方式，模型不能只返回一段难以判断的文字。下面提供一个 `submit_weekly_report` 工具，要求模型同时说明依据的记忆编号、采用的格式和三条周报内容。

In [5]:
# memory_id 让最终结果可以追溯到具体的检索记录
# format 与 items 把周报形式变成可以直接比较的字段
tools = [{
    "type": "function",
    "function": {
        "name": "submit_weekly_report",
        "description": "提交按照当前用户偏好整理的周报",
        "parameters": {
            "type": "object",
            "properties": {
                "memory_id": {"type": "string"},
                "format": {"type": "string", "enum": ["three_bullets", "table"]},
                "items": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["memory_id", "format", "items"],
        },
    },
}]

print("工具名称：", tools[0]["function"]["name"])
print("必填字段：", tools[0]["function"]["parameters"]["required"])

工具名称： submit_weekly_report
必填字段： ['memory_id', 'format', 'items']


输出说明模型只能通过同一个工具提交结果，并且必须给出三个可比较字段。工具只是统一结果格式，不负责替模型选择记忆；下一步固定唯一的正确答案。

## 2.5 定义成功标准
当前有效记录是 `M-002`，它要求使用三个简短要点。后续无论使用哪种检索方式，都必须引用这条记忆、选择对应格式，并保留三项原始事实，才算完成任务。

In [6]:
# expected 固定后续所有实验共同使用的唯一正确结果
# items 直接复用原始事实，避免检索方式改变任务内容
expected = {
    "memory_id": "M-002",
    "format": "three_bullets",
    "items": facts,
}

print(expected)

{'memory_id': 'M-002', 'format': 'three_bullets', 'items': ['接口开发完成', '自动测试通过', '使用文档待补']}


输出给出了唯一成功标准。至此，有冲突的记忆库、五种检索方式、用户任务、模型输出格式和正确答案都已固定；下一章将发送真实 API 请求，并保存后续对照共同使用的模型结果。

# 3. 获取并验证 API 响应
## 3.1 准备模型可见的记忆
先固定一条未经状态治理的召回结果 `M-001`。它包含最多的“周报偏好”原词，因此生产中的简单关键词检索很容易把它排在第一位，但它其实已经过期。本章只观察真实模型拿到这条记忆后会做什么，暂不判断结果是否正确。

In [7]:
# 这里只固定上游交给模型的召回结果，不在本章实现检索算法
# 模型只能看到编号和正文，看不到记录已经失效这一事实
recalled_memory = memory_records[0]
visible_memory = f"{recalled_memory['id']}：{recalled_memory['content']}"

print("模型可见记忆：", visible_memory)

模型可见记忆： M-001：当前周报偏好：使用表格，详细列出进展、风险和下一步。


输出显示模型只会看到 `M-001` 及其正文，不会看到 `superseded` 状态。对模型而言，这就是当前唯一可用的外部事实；下一步把它与第 2 章的同一任务一起发送给真实 API。

## 3.2 发送真实 API 请求
下面要求模型只依据可见记忆选择周报格式，并通过第 2 章定义的工具提交结果。请求使用 `.env` 中的真实 provider 和模型，同时记录实际等待时间。

In [8]:
from time import perf_counter

# system 指令限制模型只能依据本次真正注入的记忆作答
# user 消息保持第 2 章固定的任务和三项原始事实不变
messages = [
    {
        "role": "system",
        "content": "只依据提供的记忆决定格式，并调用 submit_weekly_report。memory_id 必须来自可见记忆。",
    },
    {
        "role": "user",
        "content": f"可见记忆：{visible_memory}\n请求：{user_query}\n三项进展：{'；'.join(facts)}",
    },
]

request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)
print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实 API 已经返回，完整响应保存在 `response` 中。模型此时只提交了工具调用，还没有判断它是否符合当前有效偏好；下一步读取工具名称和具体参数。

## 3.3 查看并保存模型决定
工具调用把模型决定变成了明确字段。下面取出记忆编号、周报格式和三条内容，后续章节会直接使用这份真实结果，而不是根据自然语言猜测模型意图。

In [9]:
import json

# 第一条 choice 是本次真实请求返回的模型决定
# arguments 是工具调用中的 JSON 字符串，需要还原成字典
choice = response.choices[0]
tool_call = choice.message.tool_calls[0]
model_result = json.loads(tool_call.function.arguments)

print("工具：", tool_call.function.name)
print("模型结果：", model_result)

工具： submit_weekly_report
模型结果： {'memory_id': 'M-001', 'format': 'table', 'items': ['接口开发完成', '自动测试通过', '使用文档待补']}


输出展示了真实模型依据 `M-001` 提交的结构化周报。这里只记录模型行为，不拿它与正确答案比较；下一步补齐 provider、Token、成本、延迟和停止原因。

## 3.4 查看本次请求信息
模型正常返回不等于任务正确，但真实运行信息仍必须保留。下面读取本次响应的 Token 和停止原因，并与 provider、模型和实测等待时间放在一起。

In [10]:
# usage 来自真实 API 响应，不使用字符数估算 Token
# provider 没有返回计费金额，因此成本保持为未知值
usage = response.usage
api_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": usage.prompt_tokens,
    "output_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "cost_usd": None,
    "latency_ms": api_latency_ms,
    "stop_reason": choice.finish_reason,
}

print(api_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 265, 'output_tokens': 271, 'total_tokens': 536, 'cost_usd': None, 'latency_ms': 8257, 'stop_reason': 'tool_calls'}


输出记录了本次真实调用的来源、Token、延迟和停止原因；API 没有直接返回金额，因此成本明确记为 `None`。停止原因只表示模型已经提交工具调用，不表示记忆正确或任务成功。下一章将定义直接采用单条召回结果的基线组件。

# 4. 定义基线组件
## 只按字面重合排序
生产中常见的错误基线是：记忆记录虽然保存了用户、有效状态和更新时间，检索时却只比较查询与正文是否相似。下面用相邻双字的重合数量表示最简单的文本相似度；分数相同时保留更早写入的记录。这个组件能找到字面接近的内容，却不知道记录是否属于当前用户、是否已经失效、是否已被新记录替代。

In [11]:
# 先把查询拆成相邻双字，例如“周报”和“偏好”
# 排序只读取 content，故意忽略 owner、status 和 updated_at
def retrieve_by_text_overlap(records, query):
    query_pairs = []
    for index in range(len(query) - 1):
        pair = query[index:index + 2]
        query_pairs.append(pair)

    best_memory = None
    best_score = -1
    for memory in records:
        score = 0
        for pair in query_pairs:
            if pair in memory["content"]:
                score += 1

        # 只在分数更高时替换，因此平分时会保留更早的旧记录
        if score > best_score:
            best_memory = memory
            best_score = score

    return best_memory, best_score

print("基线检索器已定义")

基线检索器已定义


输出说明基线组件已经定义，但还没有检索任何记录。它只认识正文重合分数，记录中的用户、状态和时间不会影响结果；下一章将运行这个组件，并用第 2 章的统一标准判断真实模型结果。

# 5. 展示基线故障
## 5.1 查看每条记忆的字面分数
先把每条记录单独交给基线检索器，直接查看查询与正文的重合分数。这样可以看清排序依据，而不是只看到最终选中的编号。

In [12]:
# 每次只传入一条记录，得到它与同一查询的字面分数
# 分数列表保留 id、状态和 owner，方便观察排序遗漏了什么
baseline_scores = []
for memory in memory_records:
    _, score = retrieve_by_text_overlap([memory], user_query)
    score_row = {
        "id": memory["id"],
        "score": score,
        "status": memory["status"],
        "owner": memory["owner"],
    }
    baseline_scores.append(score_row)
    print(score_row)

{'id': 'M-001', 'score': 4, 'status': 'superseded', 'owner': 'user-a'}
{'id': 'M-002', 'score': 0, 'status': 'active', 'owner': 'user-a'}
{'id': 'M-003', 'score': 0, 'status': 'active', 'owner': 'user-a'}
{'id': 'M-004', 'score': 3, 'status': 'active', 'owner': 'user-b'}


输出显示已失效的 `M-001` 得分最高，因为它比当前记录包含更多与查询重复的字词；另一名用户的 `M-004` 也得到较高分。基线分数无法表达记录是否有效、是否属于当前用户，下一步将按分数选出第一名。

## 5.2 运行基线检索
现在把完整记忆库交给基线检索器。除了打印最终编号和状态，还会确认它是否与第 3 章真正注入模型的记忆一致。

In [13]:
# 基线按照字面分数从完整记忆库中只取第一条
# 与模型结果比较可以连起“检索决定到模型行为”的因果链
baseline_memory, baseline_score = retrieve_by_text_overlap(memory_records, user_query)
baseline_matches_response = baseline_memory["id"] == model_result["memory_id"]

print("检索编号：", baseline_memory["id"])
print("记忆状态：", baseline_memory["status"])
print("字面分数：", baseline_score)
print("与真实模型引用一致：", baseline_matches_response)

检索编号： M-001
记忆状态： superseded
字面分数： 4
与真实模型引用一致： True


输出显示基线选中了已经失效的 `M-001`，第 3 章真实模型也确实引用了它。模型没有凭空选错，而是忠实使用了 Memory harness 提供的错误上下文；下一步定义两条执行路径共用的评分方式。

## 5.3 定义统一评分方式
任务成功需要同时满足三点：引用当前有效记忆、选择正确格式、保留三项原始事实。下面把这三个条件写成一个简单函数，后续改进版本继续使用同一把尺子。

In [14]:
# 记忆编号和格式必须与第 2 章固定的正确答案一致
# 每项原始事实可以扩写，但不能从最终周报中消失
def grade_report(result, target, source_facts):
    correct_memory = result["memory_id"] == target["memory_id"]
    correct_format = result["format"] == target["format"]
    facts_preserved = True

    for fact in source_facts:
        fact_found = False
        for item in result["items"]:
            if fact in item:
                fact_found = True
        if not fact_found:
            facts_preserved = False

    return {
        "correct_memory": correct_memory,
        "correct_format": correct_format,
        "facts_preserved": facts_preserved,
        "passed": correct_memory and correct_format and facts_preserved,
    }

print("统一评分函数已定义")

统一评分函数已定义


输出说明评分函数已经准备好，但还没有判断任何结果。它只检查第 2 章规定的任务目标，不关心结果来自哪种检索方法；下一步用它评价基线。

## 5.4 判断基线结果
最后把第 3 章保存的真实模型结果交给统一评分函数。模型使用了基线提供的过期记忆，因此即使三项工作事实仍然存在，完整任务也不应通过。

In [15]:
# 评分输入是本次真实 API 的结构化工具参数
# expected 与 facts 都来自第 2 章，没有为基线单独放宽标准
baseline_grade = grade_report(model_result, expected, facts)

for name, passed in baseline_grade.items():
    print(name, "：", passed)

correct_memory ： False
correct_format ： False
facts_preserved ： True
passed ： False


输出中的 `passed` 为 `False`。三项工作事实没有丢失，但基线召回了过期记忆，导致真实模型引用错误编号并采用旧格式。问题来自只看正文相似度的 Memory harness，而不是模型没有执行可见指令；下一章将定义能够处理作用域、有效状态和多种检索信号的改进组件。

# 6. 定义改进组件
## 6.1 用 Glob 缩小路径范围
可靠检索先缩小候选范围，再计算相似度。Glob 使用 `*` 等通配符匹配记忆路径，例如 `memory/user-a/preferences/*.md` 只保留当前用户的偏好文件；它速度快、结果明确，但不会理解正文含义。

In [16]:
from fnmatch import fnmatch

# pattern 描述允许进入下一阶段的路径范围
# fnmatch 只比较路径，不读取或猜测记忆正文
def glob_retrieve(records, pattern):
    matches = []
    for memory in records:
        if fnmatch(memory["path"], pattern):
            matches.append(memory)
    return matches

print("Glob 检索器已定义")

Glob 检索器已定义


输出说明 Glob 组件已经定义，但尚未读取记忆。它负责按路径收窄候选，不负责判断哪条正文最相关；下一步准备 BM25 使用的中文双字切分。

## 6.2 把中文文本切成双字词元
BM25 需要先把文本拆成可计数的词元。为了避免引入复杂分词器，这里沿用基线容易理解的相邻双字方式；例如“周报偏好”会被拆成“周报、报偏、偏好”。

In [17]:
# 每次向右移动一个字符，保留当前字符和下一个字符
# 返回的词元列表会同时用于查询和所有候选记忆
def to_bigrams(text):
    tokens = []
    for index in range(len(text) - 1):
        token = text[index:index + 2]
        tokens.append(token)
    return tokens

print("查询词元：", to_bigrams(user_query))

查询词元： ['请按', '按我', '我现', '现在', '在的', '的周', '周报', '报偏', '偏好', '好整', '整理', '理这', '这三', '三项', '项进', '进展', '展。']


输出直接展示了查询如何变成双字词元。后面的 BM25 不会读取整句含义，而是使用这些词元计算关键词分数；下一步定义完整 BM25 排序。

## 6.3 定义 BM25 检索
BM25 是搜索系统常用的稀疏检索方法：查询词在某条记录中出现越多，分数通常越高；在全部记录中很少出现的词权重更大；过长正文还会受到长度归一化。它擅长精确关键词，但难以识别“周报”和“汇报”这类不同写法表达的相近含义。

In [18]:
from rank_bm25 import BM25Okapi

# 索引只保存候选正文的双字词元，不使用记录状态或时间
# 返回列表按 BM25 分数从高到低排列，并保留原始记忆
def bm25_retrieve(records, query):
    tokenized_memories = []
    for memory in records:
        tokens = to_bigrams(memory["content"])
        tokenized_memories.append(tokens)

    index = BM25Okapi(tokenized_memories)
    scores = index.get_scores(to_bigrams(query))
    results = []
    for memory, score in zip(records, scores):
        results.append({"memory": memory, "bm25_score": float(score)})

    results.sort(key=lambda item: item["bm25_score"], reverse=True)
    return results

print("BM25 检索器已定义")

BM25 检索器已定义


输出说明 BM25 组件已经定义，但尚未建立本次任务的索引。调用 `bm25_retrieve(候选记录, 查询)` 就会得到带分数的排序结果；下一步加载真正理解语义接近程度的句向量模型。

## 6.4 加载中文嵌入模型
嵌入模型会把一句话转换成固定长度的数字向量，语义越接近，向量方向通常越接近。这里使用公开中文句向量模型 `BAAI/bge-small-zh-v1.5`；它只负责检索表示，Agent 的判断仍由 `.env` 配置的真实 API 模型完成。

In [19]:
from sentence_transformers import SentenceTransformer

# 模型名称固定，保证查询和记忆使用同一套向量空间
# 首次运行会下载公开权重，之后直接使用本地缓存
embedding_model_name = "BAAI/bge-small-zh-v1.5"
embedding_model = SentenceTransformer(embedding_model_name)
embedding_dimension = embedding_model.get_embedding_dimension()

print("嵌入模型：", embedding_model_name)
print("向量维度：", embedding_dimension)

嵌入模型： BAAI/bge-small-zh-v1.5
向量维度： 512


输出显示真实嵌入模型及其向量维度。固定长度意味着不同句子可以用同一种方式比较；下一步定义余弦相似度检索，把查询向量与每条记忆向量进行比较。

## 6.5 定义嵌入向量相似度检索
查询和候选正文经过归一化后，两个向量的点积就是余弦相似度。分数越大，表示语义越接近；这种方法能召回不同措辞的相关内容，但相似度本身仍不知道一条记忆是否过期。

In [20]:
# normalize_embeddings 让向量点积可以直接表示余弦相似度
# 每个结果同时保留原记忆和语义分数，供混合检索继续使用
def embedding_retrieve(records, query):
    contents = []
    for memory in records:
        contents.append(memory["content"])

    memory_vectors = embedding_model.encode(contents, normalize_embeddings=True)
    query_vector = embedding_model.encode(query, normalize_embeddings=True)
    scores = memory_vectors @ query_vector
    results = []
    for memory, score in zip(records, scores):
        results.append({"memory": memory, "embedding_score": float(score)})

    results.sort(key=lambda item: item["embedding_score"], reverse=True)
    return results

print("嵌入向量检索器已定义")

嵌入向量检索器已定义


输出说明嵌入向量检索器已经定义。调用 `embedding_retrieve(候选记录, 查询)` 会生成、比较并排序真实句向量；下一步把关键词排序与语义排序合并。

## 6.6 定义混合检索
BM25 分数与余弦分数不在同一量纲，直接相加会让权重难以解释。这里使用 Reciprocal Rank Fusion：只看一条记录在两份结果中的名次，再把 `1 / (60 + 名次)` 相加。它能同时利用精确关键词和语义召回，是当前搜索与 RAG 系统常用的稳健组合方式。

In [21]:
# 两种检索只贡献名次，避免直接混合不可比较的原始分数
# 同一条记忆会累加两份排名贡献，最终按 hybrid_score 排序
def hybrid_retrieve(bm25_results, embedding_results):
    scores = {}
    memories = {}

    for rank, item in enumerate(bm25_results, start=1):
        memory_id = item["memory"]["id"]
        memories[memory_id] = item["memory"]
        scores[memory_id] = scores.get(memory_id, 0) + 1 / (60 + rank)

    for rank, item in enumerate(embedding_results, start=1):
        memory_id = item["memory"]["id"]
        memories[memory_id] = item["memory"]
        scores[memory_id] = scores.get(memory_id, 0) + 1 / (60 + rank)

    results = []
    for memory_id, score in scores.items():
        results.append({"memory": memories[memory_id], "hybrid_score": score})

    results.sort(key=lambda item: item["hybrid_score"], reverse=True)
    return results

print("混合检索器已定义")

混合检索器已定义


输出说明混合检索器已经定义。它提高相关记录进入候选集的机会，但仍然只处理相关性；下一步加入最终重排序，让有效状态和更新时间拥有高于相似度的约束力。

## 6.7 定义最终重排序
相似度只能回答“内容像不像”，不能回答“事实现在能不能用”。最终重排序先移除非当前用户和已失效记录，再按混合分数排序，同分时优先较新的记录。生产系统也常把权限、状态和时间等硬约束放在语义分数之上，避免过期内容因为措辞更像查询而胜出。

In [22]:
# owner 和 status 是必须满足的硬条件，不参与模糊加权
# 通过硬条件的记录按混合分数排序，同分时选择更新时间更晚者
def rerank_memories(hybrid_results, owner):
    eligible_results = []
    for item in hybrid_results:
        memory = item["memory"]
        if memory["owner"] == owner and memory["status"] == "active":
            eligible_results.append(item)

    eligible_results.sort(
        key=lambda item: (item["hybrid_score"], item["memory"]["updated_at"]),
        reverse=True,
    )
    return eligible_results

print("最终重排序器已定义")

最终重排序器已定义


输出说明完整改进链已经准备好，但本章尚未运行检索。各组件边界清楚：Glob 管路径范围，BM25 管关键词，嵌入向量管语义，混合检索合并召回，最终重排序执行用户、状态和时间约束；下一章将逐步运行并展示每一阶段的实际结果。

# 7. 展示修复结果
## 7.1 运行 Glob 路径筛选
先用当前用户的偏好路径筛选候选。Glob 不理解正文，也不决定最终答案；它只负责把另一名用户的记录挡在后续检索之外。

In [23]:
# 路径模式只允许 user-a 的 preferences 目录进入候选集
# 单独记录耗时，便于第 8 章比较各检索阶段
glob_pattern = "memory/user-a/preferences/*.md"
glob_started = perf_counter()
glob_results = glob_retrieve(memory_records, glob_pattern)
glob_latency_ms = round((perf_counter() - glob_started) * 1000, 3)

glob_ids = []
for memory in glob_results:
    glob_ids.append(memory["id"])

print("路径模式：", glob_pattern)
print("候选记录：", glob_ids)
print("耗时：", glob_latency_ms, "ms")

路径模式： memory/user-a/preferences/*.md
候选记录： ['M-001', 'M-002', 'M-003']
耗时： 0.139 ms


输出显示 `M-004` 已被路径范围排除，候选从四条缩小到当前用户的三条。`M-001` 仍在其中，因为 Glob 不读取有效状态；下一步用 BM25 对三条正文排序。

## 7.2 运行 BM25 检索
BM25 会根据第 6 章展示的双字词元计算关键词分数。这里完整打印三条候选的排名，观察精确关键词是否足以解决新旧冲突。

In [24]:
# BM25 只处理 Glob 留下的当前用户候选，不再搜索全部记忆
# 每条分数来自同一查询，数值越大表示关键词匹配越强
bm25_started = perf_counter()
bm25_results = bm25_retrieve(glob_results, user_query)
bm25_latency_ms = round((perf_counter() - bm25_started) * 1000, 3)

for item in bm25_results:
    print(item["memory"]["id"], "|", round(item["bm25_score"], 4))
print("耗时：", bm25_latency_ms, "ms")

M-001 | 1.8671
M-002 | 0.0
M-003 | 0.0
耗时： 0.271 ms


输出显示 BM25 把 `M-001` 排在第一位，因为旧记录包含更多与查询完全相同的字词。关键词检索找到了字面上最像的记录，却仍不知道它已经失效；下一步观察语义向量能否单独解决问题。

## 7.3 运行嵌入向量相似度检索
现在用真实中文嵌入模型比较查询和三条候选的语义。余弦分数越接近 `1`，表示向量方向越接近；它可以理解不同措辞，但仍只判断内容相关性。

In [25]:
# 查询和三条候选都由同一个 BGE 模型生成归一化向量
# 耗时包含本次实际编码与余弦相似度排序
embedding_started = perf_counter()
embedding_results = embedding_retrieve(glob_results, user_query)
embedding_latency_ms = round((perf_counter() - embedding_started) * 1000, 3)

for item in embedding_results:
    print(item["memory"]["id"], "|", round(item["embedding_score"], 4))
print("耗时：", embedding_latency_ms, "ms")

M-001 | 0.8035
M-002 | 0.6654
M-003 | 0.499
耗时： 272.121 ms


输出显示嵌入向量仍把 `M-001` 排在第一位。旧记录在语义上确实更像“周报偏好”，所以更强的语义模型也不会自动知道它已经过期；下一步合并 BM25 与向量排名。

## 7.4 运行混合检索
混合检索通过 RRF 合并 BM25 和嵌入向量的名次。它能减少单一检索信号漏掉相关内容的风险，但如果两种方法都偏爱同一条过期记录，混合本身仍无法判断事实是否有效。

In [26]:
# 输入直接复用刚才真实运行得到的两份排名
# RRF 只合并名次，不重复计算 BM25 或句向量
hybrid_started = perf_counter()
hybrid_results = hybrid_retrieve(bm25_results, embedding_results)
hybrid_latency_ms = round((perf_counter() - hybrid_started) * 1000, 3)

for item in hybrid_results:
    print(item["memory"]["id"], "|", round(item["hybrid_score"], 6))
print("耗时：", hybrid_latency_ms, "ms")

M-001 | 0.032787
M-002 | 0.032258
M-003 | 0.031746
耗时： 0.035 ms


输出显示混合检索仍把 `M-001` 排在第一位，因为 BM25 和嵌入向量都认为它最相关。相关性召回已经完成，但新旧冲突还没有解决；下一步执行最终重排序。

## 7.5 运行最终重排序
最终重排序把用户和有效状态当作硬条件，先移除过期记录，再保留混合相关性顺序。这样，相似度再高的旧记忆也不能进入模型上下文。

In [27]:
# owner 和 active 状态由 Memory harness 明确执行，不依赖模型猜测
# 输出保留混合分数，便于观察硬约束前后的排名变化
rerank_started = perf_counter()
reranked_results = rerank_memories(hybrid_results, current_user)
rerank_latency_ms = round((perf_counter() - rerank_started) * 1000, 3)

for item in reranked_results:
    memory = item["memory"]
    print(memory["id"], "|", memory["status"], "|", round(item["hybrid_score"], 6))
print("耗时：", rerank_latency_ms, "ms")

M-002 | active | 0.032258
M-003 | active | 0.031746
耗时： 0.027 ms


输出显示过期的 `M-001` 已被移除，当前有效的 `M-002` 成为第一名，无关的 `M-003` 保留在后面。检索链现在给出了正确记忆；下一步只把第一名注入真实模型。

## 7.6 组装修复后的模型上下文
改进版本只注入重排序后的第一条记录，不把整个记忆库塞进上下文。任务、三项进展、工具和系统指令都与第 3 章保持相同，唯一变化是 Memory harness 选择的记忆。

In [28]:
# 最终上下文只使用重排序第一名，避免把冲突记忆一同交给模型
# 消息结构与第 3 章相同，使前后实验只改变记忆内容
fixed_memory = reranked_results[0]["memory"]
fixed_visible_memory = f"{fixed_memory['id']}：{fixed_memory['content']}"
fixed_messages = [
    {
        "role": "system",
        "content": "只依据提供的记忆决定格式，并调用 submit_weekly_report。memory_id 必须来自可见记忆。",
    },
    {
        "role": "user",
        "content": f"可见记忆：{fixed_visible_memory}\n请求：{user_query}\n三项进展：{'；'.join(facts)}",
    },
]

print("修复后可见记忆：", fixed_visible_memory)

修复后可见记忆： M-002：后来确认：汇报只保留三个简短要点，不要表格。


输出显示真实模型这次只会看到有效记录 `M-002`。相比第 3 章，模型、任务和工具都没有变化；下一步发送第二次真实 API 请求。

## 7.7 获取修复后的真实响应
现在把修复后的上下文发送给 `.env` 指定的同一真实模型，并记录实际等待时间。模型仍必须通过同一个 `submit_weekly_report` 工具提交结果。

In [29]:
# 请求参数与基线保持一致，只替换 messages 中的可见记忆
# 单独保存响应和延迟，避免覆盖第 3 章的基线证据
fixed_request_started = perf_counter()
fixed_response = client.chat.completions.create(
    model=model_name,
    messages=fixed_messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
fixed_api_latency_ms = round((perf_counter() - fixed_request_started) * 1000)

print("修复后的真实 API 响应已收到")

修复后的真实 API 响应已收到


输出说明同一真实模型已经根据修复上下文返回工具调用。此时只知道通信完成，还不能宣称任务成功；下一步读取结构化结果。

## 7.8 查看修复后的模型决定
下面按照与第 3 章相同的方式读取工具名称和 JSON 参数。结果会明确显示模型引用了哪条记忆、选择了什么格式，以及保留了哪些事实。

In [30]:
# fixed_choice 保存改进请求的第一条真实模型决定
# fixed_result 是后续统一评分直接读取的结构化字典
fixed_choice = fixed_response.choices[0]
fixed_tool_call = fixed_choice.message.tool_calls[0]
fixed_result = json.loads(fixed_tool_call.function.arguments)

print("工具：", fixed_tool_call.function.name)
print("修复后结果：", fixed_result)

工具： submit_weekly_report
修复后结果： {'memory_id': 'M-002', 'format': 'three_bullets', 'items': ['接口开发完成', '自动测试通过', '使用文档待补']}


输出显示真实模型引用 `M-002`，采用三个要点格式，并保留三项工作事实。下一步记录这次请求的 Token、成本、延迟和停止原因。

## 7.9 查看修复请求信息
改进版本增加了本地检索计算，但模型请求仍会产生真实 Token 和等待时间。下面用与基线完全相同的字段保存本次 API 信息。

In [31]:
# Token 与停止原因直接来自修复后的真实 API 响应
# provider 未返回金额，因此成本继续使用相同的未知值
fixed_usage = fixed_response.usage
fixed_api_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": fixed_usage.prompt_tokens,
    "output_tokens": fixed_usage.completion_tokens,
    "total_tokens": fixed_usage.total_tokens,
    "cost_usd": None,
    "latency_ms": fixed_api_latency_ms,
    "stop_reason": fixed_choice.finish_reason,
}

print(fixed_api_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 263, 'output_tokens': 230, 'total_tokens': 493, 'cost_usd': None, 'latency_ms': 14173, 'stop_reason': 'tool_calls'}


输出记录了修复请求的真实运行信息。停止原因仍只表示模型提交了工具调用；下一步使用第 5 章的同一个评分函数判断任务是否真正完成。

## 7.10 判断修复结果
模型、任务、工具和评分标准都没有变化。下面只把修复后的真实工具参数交给统一评分函数，检查记忆编号、格式和三项事实。

In [32]:
# 评分继续使用第 2 章的 expected 和原始 facts
# 与基线共用 grade_report，避免改进版本使用更宽松标准
fixed_grade = grade_report(fixed_result, expected, facts)

for name, passed in fixed_grade.items():
    print(name, "：", passed)

correct_memory ： True
correct_format ： True
facts_preserved ： True
passed ： True


输出中的四项结果全部为 `True`。真实模型在拿到当前有效记忆后完成了任务，说明修复来自外层检索与生命周期约束，而不是更换模型或放宽标准；下一章汇总各检索阶段和端到端消融。

# 8. 汇总消融对照
## 8.1 补齐基线检索耗时
第 5 章已经保存了基线选择和任务结果，但没有计时。下面在同一内存数据上重新运行一次相同基线函数，只补充检索耗时，不重新调用模型。

In [33]:
# 计时只包住第 4 章定义的同一个字面重合检索函数
# 检索结果仍应是 M-001，本格不会产生新的模型费用
baseline_retrieval_started = perf_counter()
measured_baseline_memory, _ = retrieve_by_text_overlap(memory_records, user_query)
baseline_retrieval_latency_ms = round(
    (perf_counter() - baseline_retrieval_started) * 1000,
    3,
)

print("基线首位：", measured_baseline_memory["id"])
print("基线检索耗时：", baseline_retrieval_latency_ms, "ms")

基线首位： M-001
基线检索耗时： 0.072 ms


输出再次确认基线首位是 `M-001`，并补齐了本次实测检索耗时。下一步把基线、Glob、BM25、嵌入向量、混合检索和重排序放进同一张表。

## 8.2 对比各检索阶段
这张表关注每种方法实际改变了什么：候选数量、排序第一名、第一名是否可用，以及本次运行耗时。Glob 只筛选不排序，因此不把候选列表的第一项当成检索结论。

In [34]:
import pandas as pd

# 每行使用前面已经运行并保存的真实候选、排名和耗时
# task_ready 表示该阶段第一名能否直接用于当前任务
retrieval_rows = [
    {"method": "字面重合基线", "candidates": 4, "top_memory": baseline_memory["id"], "task_ready": False, "latency_ms": baseline_retrieval_latency_ms},
    {"method": "Glob", "candidates": len(glob_results), "top_memory": "未排序", "task_ready": False, "latency_ms": glob_latency_ms},
    {"method": "BM25", "candidates": len(bm25_results), "top_memory": bm25_results[0]["memory"]["id"], "task_ready": False, "latency_ms": bm25_latency_ms},
    {"method": "嵌入向量相似度", "candidates": len(embedding_results), "top_memory": embedding_results[0]["memory"]["id"], "task_ready": False, "latency_ms": embedding_latency_ms},
    {"method": "混合检索", "candidates": len(hybrid_results), "top_memory": hybrid_results[0]["memory"]["id"], "task_ready": False, "latency_ms": hybrid_latency_ms},
    {"method": "最终重排序", "candidates": len(reranked_results), "top_memory": reranked_results[0]["memory"]["id"], "task_ready": True, "latency_ms": rerank_latency_ms},
]
retrieval_comparison = pd.DataFrame(retrieval_rows)

display(retrieval_comparison)

,method,candidates,top_memory,task_ready,latency_ms
0,字面重合基线,4,M-001,False,0.072
1,Glob,3,未排序,False,0.139
2,BM25,3,M-001,False,0.271
3,嵌入向量相似度,3,M-001,False,272.121
4,混合检索,3,M-001,False,0.035
5,最终重排序,2,M-002,True,0.027


表格显示 Glob 移除了跨用户记录，但 BM25、嵌入向量和混合检索仍把过期 `M-001` 排在第一位。只有最终重排序执行有效状态约束后，首位才变为 `M-002`。不同耗时是本次本机实测值；可以直接确认的机制效果是排序方向，而不是固定速度收益。

## 8.3 对比端到端结果
最后比较两条完整路径。两者使用同一个真实模型、任务、工具和评分函数；改进版本额外付出本地检索时间，目标是把任务成功率从失败变为成功。

In [35]:
# 改进检索耗时是五个阶段在本次运行中的实测值之和
# 总耗时同时包含本地检索和各自真实 API 等待时间
improved_retrieval_latency_ms = round(
    glob_latency_ms + bm25_latency_ms + embedding_latency_ms + hybrid_latency_ms + rerank_latency_ms,
    3,
)
ablation_rows = [
    {
        "variant": "错误基线",
        "memory": baseline_memory["id"],
        "format": model_result["format"],
        "success_rate": "0%",
        "retrieval_ms": baseline_retrieval_latency_ms,
        "api_ms": api_metrics["latency_ms"],
        "total_ms": round(baseline_retrieval_latency_ms + api_metrics["latency_ms"], 3),
        "tokens": api_metrics["total_tokens"],
        "cost_usd": api_metrics["cost_usd"],
    },
    {
        "variant": "改进版本",
        "memory": fixed_memory["id"],
        "format": fixed_result["format"],
        "success_rate": "100%",
        "retrieval_ms": improved_retrieval_latency_ms,
        "api_ms": fixed_api_metrics["latency_ms"],
        "total_ms": round(improved_retrieval_latency_ms + fixed_api_metrics["latency_ms"], 3),
        "tokens": fixed_api_metrics["total_tokens"],
        "cost_usd": fixed_api_metrics["cost_usd"],
    },
]
ablation_comparison = pd.DataFrame(ablation_rows)

display(ablation_comparison)

,variant,memory,format,success_rate,retrieval_ms,api_ms,total_ms,tokens,cost_usd
0,错误基线,M-001,table,0%,0.072,8257,8257.072,536,None
1,改进版本,M-002,three_bullets,100%,272.593,14173,14445.593,493,None


表格显示，基线引用 `M-001` 并失败，改进版本引用 `M-002` 并成功，单任务成功率从 `0%` 变为 `100%`。改进版本增加了本地检索计算；API Token 和延迟来自两次真实调用，金额因 provider 未返回而保持未知。下一步用一个状态变化收束因果关系。

## 8.4 总结机制效果
下面只保留加入完整 Memory 检索链前后的关键状态，避免用大量指标掩盖真正变化：模型没有更换，改变的是它在决策前看到的记忆。

In [36]:
# 状态变化连接了检索选择、模型格式与任务评分三个层次
# 两端都来自本次从头执行保存的真实 API 与统一评分结果
memory_effect = {
    "retrieved_memory": f"{baseline_memory['id']} -> {fixed_memory['id']}",
    "report_format": f"{model_result['format']} -> {fixed_result['format']}",
    "task_success": f"{baseline_grade['passed']} -> {fixed_grade['passed']}",
}

for name, change in memory_effect.items():
    print(name, "：", change)

retrieved_memory ： M-001 -> M-002
report_format ： table -> three_bullets
task_success ： False -> True


输出中的 `M-001 -> M-002`、`table -> three_bullets` 和 `False -> True` 说明：大模型两次都忠实执行了可见记忆，真正决定任务成败的是外层 Memory harness 是否把作用域、相关性、新近性和有效状态组合成完整检索链。至此，本 Notebook 的消融对照结束。

## 8.5 拓展

### nano 版省略了什么

nano 版只在小型内存集合上演示 Glob、BM25、向量与重排，没有持久存储、写入策略、遗忘、去重、时效衰减、隐私边界、跨用户隔离、反馈学习和大规模索引运维。生产 Memory 还需分别评估写得对、找得到、用得对和该忘时能忘，而不能只看检索相似度。

### 延伸阅读


1. 2025, [Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory](https://arxiv.org/abs/2504.19413)：长期记忆抽取、更新与生产化评测。
2. 2026, [LongMemEval-V2](https://arxiv.org/abs/2605.12493)：面向长期 Agent Memory 的经验积累与跨会话能力评估。
3. 2026, [Memory for Autonomous LLM Agents](https://arxiv.org/abs/2603.07670)：记忆机制、评测方法与新兴研究方向综述。